<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->


# Cosmos3 Video Transfer with TensorRT-LLM

Run the five established controls—Canny/edge, blur, depth, segmentation, and WSM—against an
already-running TensorRT-LLM VisualGen server. The same notebook works with Cosmos3-Nano or
Cosmos3-Super; select the checkpoint when launching the server.


## Start the TensorRT-LLM server

Follow the shared [TensorRT-LLM setup](../../README.md#tensorrt-llm-generator), then launch one
server from the TensorRT-LLM checkout:

```bash
export TRTLLM_ROOT="${TRTLLM_ROOT:-$PWD}"

# Nano: one GPU
trtllm-serve nvidia/Cosmos3-Nano \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml" \
  --port 8000

# Super: four GPUs (run instead of Nano)
torchrun --nproc_per_node=4 -m tensorrt_llm.commands.serve \
  nvidia/Cosmos3-Super \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-super-4gpu.yaml" \
  --port 8000
```

Install `requests` in this notebook kernel if needed. The response is synchronous encoded video
bytes. Inputs use multipart `input_reference`; precomputed depth/seg/WSM controls are base64 media
inside JSON `extra_params`, because JSON has no byte type. TensorRT-LLM decodes them back to bytes
at the HTTP boundary.


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
TRANSFER_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "transfer"
OUTPUT_ROOT = Path(
    os.environ.get("COSMOS3_TRTLLM_TRANSFER_OUTPUT_ROOT", TRANSFER_ROOT / "outputs" / "notebooks" / "trt_llm")
).resolve()
TRTLLM_BASE_URL = os.environ.get("COSMOS3_TRTLLM_BASE_URL", "http://localhost:8000").rstrip("/")
TRTLLM_API_KEY = os.environ.get("COSMOS3_TRTLLM_API_KEY", "tensorrt_llm")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("COSMOS_ROOT:", COSMOS_ROOT)
print("TRANSFER_ROOT:", TRANSFER_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("TRTLLM_BASE_URL:", TRTLLM_BASE_URL)


## Request matrix

| Control | Transport | Frames / fps | Text / control guidance |
| --- | --- | ---: | ---: |
| Edge | Uploaded source; GPU Canny | 121 / 30 | 3.0 / 1.5 |
| Blur | Uploaded source; GPU bilateral blur | 121 / 30 | 3.0 / 1.5 |
| Depth | Uploaded source + base64 precomputed control | 121 / 30 | 3.0 / 1.5 |
| Segmentation | Uploaded source + base64 precomputed control | 121 / 30 | 3.0 / 2.0 |
| WSM | Uploaded source + base64 precomputed control | 101 / 10 | 1.0 / 3.0 |

All checked-in controls are 16:9 and requests explicitly select `1280x720`. TensorRT-LLM can
also derive the nearest supported bucket from an uploaded source when neither dimension is set.


In [ ]:
import base64
import json
import time

import requests
from IPython.display import Video, display


def server_root_url() -> str:
    return TRTLLM_BASE_URL[:-3] if TRTLLM_BASE_URL.endswith("/v1") else TRTLLM_BASE_URL


def video_api_url() -> str:
    root = TRTLLM_BASE_URL if TRTLLM_BASE_URL.endswith("/v1") else f"{TRTLLM_BASE_URL}/v1"
    return f"{root}/videos/generations"


def wait_for_server(timeout_s: int = 1800, interval_s: int = 10) -> None:
    deadline = time.time() + timeout_s
    url = f"{server_root_url()}/health"
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=10)
            if response.ok:
                print("TensorRT-LLM server is ready:", url)
                return
        except requests.RequestException as exc:
            print("waiting for TensorRT-LLM:", exc)
        time.sleep(interval_s)
    raise TimeoutError(f"TensorRT-LLM did not become ready at {url}")


CONTROL_CASES = {
    "edge": {
        "control": "assets/edge/control_edge.mp4",
        "prompt": "assets/edge/prompt.json",
        "num_frames": 121,
        "fps": 30,
        "guidance_scale": 3.0,
        "control_guidance": 1.5,
        "hint": {"preset_edge_threshold": "medium"},
    },
    "blur": {
        "control": "assets/blur/control_blur.mp4",
        "prompt": "assets/blur/prompt.json",
        "num_frames": 121,
        "fps": 30,
        "guidance_scale": 3.0,
        "control_guidance": 1.5,
        "hint": {"preset_blur_strength": "medium"},
    },
    "depth": {
        "control": "assets/depth/control_depth.mp4",
        "prompt": "assets/depth/prompt.json",
        "num_frames": 121,
        "fps": 30,
        "guidance_scale": 3.0,
        "control_guidance": 1.5,
    },
    "seg": {
        "control": "assets/seg/control_seg.mp4",
        "prompt": "assets/seg/prompt.json",
        "num_frames": 121,
        "fps": 30,
        "guidance_scale": 3.0,
        "control_guidance": 2.0,
    },
    "wsm": {
        "control": "assets/wsm/control_wsm.mp4",
        "prompt": "assets/wsm/prompt.json",
        "num_frames": 101,
        "fps": 10,
        "guidance_scale": 1.0,
        "control_guidance": 3.0,
    },
}


def compact_json(path: Path) -> str:
    return json.dumps(json.loads(path.read_text()), ensure_ascii=True, separators=(",", ":"))


def run_transfer(control_name: str) -> Path:
    """Upload the source as multipart media and carry precomputed controls as base64."""
    spec = CONTROL_CASES[control_name]
    control_path = (TRANSFER_ROOT / spec["control"]).resolve()
    prompt_path = (TRANSFER_ROOT / spec["prompt"]).resolve()
    negative_path = TRANSFER_ROOT / "assets" / "negative_prompt.json"
    for path in (control_path, prompt_path, negative_path):
        if not path.exists():
            raise FileNotFoundError(path)

    hint = dict(spec.get("hint", {}))
    # Edge and blur have on-the-fly GPU preprocessors, so the uploaded source is
    # sufficient. Depth, segmentation and WSM require encoded precomputed media.
    if control_name not in {"edge", "blur"}:
        hint["control"] = base64.b64encode(control_path.read_bytes()).decode("ascii")

    extra_params = {
        "use_duration_template": False,
        "use_resolution_template": False,
        "use_system_prompt": False,
        "use_guardrails": True,
        control_name: hint,
        "control_guidance": spec["control_guidance"],
        "num_video_frames_per_chunk": spec["num_frames"],
        "num_conditional_frames": 1,
        "num_first_chunk_conditional_frames": 0,
        "max_frames": spec["num_frames"],
    }
    form = {
        "prompt": compact_json(prompt_path),
        "negative_prompt": compact_json(negative_path),
        "size": "1280x720",
        "num_frames": str(spec["num_frames"]),
        "fps": str(spec["fps"]),
        "num_inference_steps": "35",
        "guidance_scale": str(spec["guidance_scale"]),
        "max_sequence_length": "4096",
        "seed": "2026",
        "format": "auto",
        "extra_params": json.dumps(extra_params, separators=(",", ":")),
    }
    headers = {"Accept": "video/mp4, video/x-msvideo"}
    if TRTLLM_API_KEY:
        headers["Authorization"] = f"Bearer {TRTLLM_API_KEY}"
    with control_path.open("rb") as source_file:
        response = requests.post(
            video_api_url(),
            data=form,
            files={"input_reference": (control_path.name, source_file, "video/mp4")},
            headers=headers,
            timeout=3600,
        )
    if not response.ok:
        raise RuntimeError(f"TensorRT-LLM request failed ({response.status_code}): {response.text}")
    content_type = response.headers.get("content-type", "").lower()
    suffix = ".avi" if "avi" in content_type or response.content[:4] == b"RIFF" else ".mp4"
    if not response.content:
        raise RuntimeError("TensorRT-LLM returned an empty media response")
    output_path = OUTPUT_ROOT / f"transfer_{control_name}{suffix}"
    output_path.write_bytes(response.content)
    print("control:", control_path)
    print("extra_params:", json.dumps({**extra_params, control_name: "<control media>"}, indent=2))
    print("saved:", output_path, content_type)
    return output_path


def view_transfer(path: Path) -> None:
    display(Video(str(path), embed=True))


## Canny / Edge transfer


In [ ]:
wait_for_server()
edge_output = run_transfer("edge")


In [ ]:
view_transfer(edge_output)


## Blur transfer


In [ ]:
wait_for_server()
blur_output = run_transfer("blur")


In [ ]:
view_transfer(blur_output)


## Depth transfer


In [ ]:
wait_for_server()
depth_output = run_transfer("depth")


In [ ]:
view_transfer(depth_output)


## Segmentation transfer


In [ ]:
wait_for_server()
seg_output = run_transfer("seg")


In [ ]:
view_transfer(seg_output)


## World Scenario Model (WSM) transfer


In [ ]:
wait_for_server()
wsm_output = run_transfer("wsm")


In [ ]:
view_transfer(wsm_output)
